In [7]:
!pip install datasets tokenizers huggingface_hub ijson -q
print("Dependencies installed.")

Dependencies installed.


In [8]:
import os, json, math, time, re, zipfile, random
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from tokenizers import ByteLevelBPETokenizer
from huggingface_hub import login, hf_hub_download
from datasets import load_dataset
import ijson

HF_TOKEN = "REDACTED_HF_TOKEN"
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU  = torch.cuda.device_count()
print(f"Device: {DEVICE}, GPUs: {N_GPU}")
if torch.cuda.is_available():
    for i in range(N_GPU):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda, GPUs: 2
GPU 0: Tesla T4 (15.6 GB)
GPU 1: Tesla T4 (15.6 GB)


In [9]:
CFG = {
    "repo"        : "Exploration-Lab/CS-781-Capstone",
    "work_dir"    : "/kaggle/working",
    "vocab_size"  : 32000,

    "n_layers"    : 16,
    "n_heads"     : 12,
    "n_kv_heads"  : 4,
    "d_model"     : 768,
    "d_ff"        : 2048,
    "ctx_len"     : 1024,
    "local_window": 256,
    "dropout"     : 0.1,
    "rope_theta"  : 10000.0,

    "wiki_tokens" : 80_000_000,
    "wiki_lr"     : 3e-4,
    "wiki_lr_min" : 3e-5,
    "wiki_warmup" : 1000,
    "wiki_hours"  : 2.0,

    "legal_tokens": 320_000_000,
    "legal_lr"    : 1e-4,
    "legal_lr_min": 1e-5,
    "legal_warmup": 500,
    "legal_hours" : 8.0,
    # Session safety limit
    "session_max_hours": 10.5,
    # Data mix
    "mix": {
        "train_supreme_court": 0.40,
        "train_high_courts"  : 0.30,
        "train_acts"         : 0.20,
        "train_tribunals"    : 0.10,
    },
    # Optimization
    "batch_size"  : 8,
    "grad_accum"  : 16,
    "weight_decay": 0.1,
    "beta1"       : 0.9,
    "beta2"       : 0.95,
    "grad_clip"   : 1.0,
    "save_every"  : 500,
    "log_every"   : 50,
    # Generation
    "temperature" : 0.80,
    "top_p"       : 0.92,
    "rep_penalty" : 1.15,
    "target_words": 128,
    "max_new_tok" : 240,
    "gen_batch"   : 16,
}

WORK     = Path(CFG["work_dir"])
CKPT_DIR = WORK / "checkpoints"
TOK_DIR  = WORK / "tokenizer"
DATA_DIR = WORK / "data"

for d in [CKPT_DIR, TOK_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SESSION_START = time.time()
print("Config initialized.")

Config initialized.


In [10]:
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
test_path = hf_hub_download(repo_id=CFG["repo"], filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
local_train = {} 
print("Test prompts loaded!")

Test prompts loaded!


In [17]:
TRAIN_FILES = {
    "train_supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "train_high_courts"  : "Phase_1_Pretraining/Train/train_high_courts.json",
    "train_acts"         : "Phase_1_Pretraining/Train/train_acts.json",
    "train_tribunals"    : "Phase_1_Pretraining/Train/train_tribunals.json",
}
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

local_train = {}
for name, path in TRAIN_FILES.items():
    print(f"Fetching {name}...")
    local_train[name] = hf_hub_download(
        repo_id=CFG["repo"], filename=path,
        repo_type="dataset", token=HF_TOKEN
    )
    sz = os.path.getsize(local_train[name]) / 1e9
    print(f"  {name}: {sz:.2f} GB")

test_path = hf_hub_download(
    repo_id=CFG["repo"], filename=TEST_FILE,
    repo_type="dataset", token=HF_TOKEN
)
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Test prompts ready: {len(test_data)} examples.")

Fetching train_supreme_court...
  train_supreme_court: 1.32 GB
Fetching train_high_courts...
  train_high_courts: 10.67 GB
Fetching train_acts...
  train_acts: 0.73 GB
Fetching train_tribunals...
  train_tribunals: 6.48 GB
Test prompts ready: 1000 examples.


In [11]:
def stream_texts(filepath, max_chars=None):
    chars = 0
    with open(filepath, "rb") as f:
        for rec in ijson.items(f, "item"):
            text = rec.get("text", "")
            if len(text) < 100:
                continue
            yield text
            chars += len(text)
            if max_chars and chars >= max_chars:
                break

def get_tokenizer(local_train, cfg):
    tok_file = TOK_DIR / "vocab.json"
    if tok_file.exists():
        print("Loading existing trained tokenizer...")
        return ByteLevelBPETokenizer(
            vocab=str(TOK_DIR / "vocab.json"),
            merges=str(TOK_DIR / "merges.txt")
        )

    corpus_file = WORK / "tok_corpus.txt"
    total_target = 20_000_000
    print("Sampling 20M characters across all 4 Indian legal sources...")
    with open(corpus_file, "w", encoding="utf-8") as out:
        for name, ratio in cfg["mix"].items():
            budget = int(total_target * ratio)
            collected = 0
            for text in stream_texts(local_train[name], max_chars=budget):
                out.write(text.strip() + "\n\n")
                collected += len(text)
            print(f"  {name}: {collected/1e6:.1f}M chars")

    print("Training ByteLevelBPE tokenizer (vocab=32000)...")
    tok = ByteLevelBPETokenizer()
    tok.train(
        files=[str(corpus_file)],
        vocab_size=cfg["vocab_size"],
        min_frequency=3,
        special_tokens=["<s>", "</s>", "<unk>", "<pad>", "<mask>"]
    )
    tok.save_model(str(TOK_DIR))
    if corpus_file.exists():
        corpus_file.unlink()
    print("Tokenizer saved successfully.")
    return tok

tokenizer = get_tokenizer(local_train, CFG)
VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"Vocabulary size: {VOCAB_SIZE}")

Loading existing trained tokenizer...
Vocabulary size: 32000


In [16]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, seq_len):
        if seq_len > self.cos_cached.shape[2]:
            self._build_cache(seq_len)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class BiScaleGQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, ctx_len, rope_theta, is_local=False, window_size=256):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_head     = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.d_head, max_seq=ctx_len, theta=rope_theta)

        causal = torch.tril(torch.ones(ctx_len, ctx_len))
        if is_local:
            causal = causal * torch.triu(torch.ones(ctx_len, ctx_len), diagonal=-window_size)
        self.register_buffer("mask", causal.view(1, 1, ctx_len, ctx_len))


        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        cos, sin = self.rope(T)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        scale = dh ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        for proj in [self.gate, self.up, self.down]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LearnedGatedResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim, bias=True)

        nn.init.normal_(self.gate.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.gate.bias, 2.0)

    def forward(self, x, sublayer):
        g = torch.sigmoid(self.gate(x))
        return x + g * sublayer

class BiScaleLegalBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        is_local = (layer_idx < (cfg["n_layers"] // 2))
        self.norm1 = RMSNorm(cfg["d_model"])
        self.attn  = BiScaleGQA(
            cfg["d_model"], cfg["n_heads"], cfg["n_kv_heads"],
            cfg["dropout"], cfg["ctx_len"], cfg["rope_theta"],
            is_local=is_local, window_size=cfg["local_window"]
        )
        self.res1 = LearnedGatedResidual(cfg["d_model"])
        self.norm2 = RMSNorm(cfg["d_model"])
        self.ffn   = SwiGLU(cfg["d_model"], cfg["d_ff"])
        self.res2  = LearnedGatedResidual(cfg["d_model"])
        self.drop  = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.res1(x, self.drop(self.attn(self.norm1(x))))
        x = self.res2(x, self.drop(self.ffn(self.norm2(x))))
        return x

class BiScaleLegalLLaMA(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.ctx     = cfg["ctx_len"]
        self.tok_emb = nn.Embedding(vocab_size, cfg["d_model"])
        self.drop    = nn.Dropout(cfg["dropout"])
        self.blocks  = nn.ModuleList([BiScaleLegalBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.norm    = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.weight


        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate_batch(self, ids, max_new, temperature=0.80, top_p=0.92, rep_penalty=1.15):
        import time
        self.eval()
        out = ids.clone()
        B = out.shape[0]
        print(f"DEBUG: Starting generation for {max_new} tokens. Batch shape: {out.shape}")
        
        for step in range(max_new):
            t0 = time.time()
            
            # Forward pass
            logits, _ = self(out[:, -self.ctx:])
            logits = logits[:, -1, :].float()
            
            # Vectorized repetition penalty
            score = torch.gather(logits, 1, out)
            score = torch.where(score < 0, score * rep_penalty, score / rep_penalty)
            logits.scatter_(1, out, score)
            
            # Sampling
            logits /= max(temperature, 1e-6)
            probs = F.softmax(logits, -1)
            sp, si = probs.sort(-1, descending=True)
            sp[sp.cumsum(-1) - sp > top_p] = 0.0
            sp /= sp.sum(-1, keepdim=True)
            next_tok = si.gather(-1, torch.multinomial(sp, 1))
            
            out = torch.cat([out, next_tok], dim=-1)
            t1 = time.time()
            
            if step % 50 == 0:
                print(f"DEBUG: Step {step}/{max_new} took {t1-t0:.4f} seconds.")
                
        self.train()
        return out

def get_raw_model(model):
    m = model
    if hasattr(m, "module"): m = m.module
    return m

model = BiScaleLegalLLaMA(CFG, VOCAB_SIZE).to(DEVICE)
if N_GPU > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel on {N_GPU} GPUs.")

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params/1e6:.2f}M")

DataParallel on 2 GPUs.
Parameters: 144.16M


In [17]:
def find_checkpoint(stage):
    search = [CKPT_DIR]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for d in kaggle_input.iterdir():
            if (d / "checkpoints").exists():
                search.append(d / "checkpoints")

    for path in search:
        ckpts = sorted(path.glob(f"stage{stage}_step_*.pt"))
        if ckpts:
            print(f"Found resume checkpoint: {ckpts[-1].name}")
            return ckpts[-1]
    return None

def stage_done(stage):
    flag = CKPT_DIR / f"stage{stage}_done.pt"
    return flag.exists()

def mark_stage_done(stage, model, val_loss, tokens):
    flag = CKPT_DIR / f"stage{stage}_done.pt"
    torch.save({
        "model": get_raw_model(model).state_dict(),
        "val_loss": val_loss,
        "tokens": tokens
    }, flag)
    print(f"Stage {stage} successfully completed (val_loss={val_loss:.4f}).")

def save_checkpoint(stage, step, tokens, val_loss, model, optimizer, scaler):
    path = CKPT_DIR / f"stage{stage}_step_{step:07d}.pt"
    torch.save({
        "step": step, "tokens": tokens, "val_loss": val_loss,
        "model": get_raw_model(model).state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
    }, path)
    old = sorted(CKPT_DIR.glob(f"stage{stage}_step_*.pt"))[:-3]
    for f in old:
        f.unlink()
    print(f"  Saved checkpoint: step={step} | tokens={tokens/1e6:.1f}M | val_loss={val_loss:.4f}")

def load_checkpoint(path, model, optimizer, scaler):
    ckpt = torch.load(path, map_location=DEVICE)
    get_raw_model(model).load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    return ckpt["step"], ckpt["tokens"], ckpt["val_loss"]

print("Checkpoint utilities ready.")

Checkpoint utilities ready.


In [ ]:
class WikiDataset(IterableDataset):
    def __init__(self, tokenizer, ctx_len, max_tokens, skip_tokens=0):
        self.tok = tokenizer
        self.ctx = ctx_len
        self.max_tokens = max_tokens
        self.skip = skip_tokens

    def __iter__(self):
        ds = load_dataset("wikimedia/wikipedia", "20231101.en",
                          split="train", streaming=True, trust_remote_code=False)
        buf = []
        yielded = 0
        skipped = 0

        for item in ds:
            text = item.get("text", "")
            if len(text) < 200:
                continue
            ids = self.tok.encode(text).ids
            buf.extend(ids)

            while len(buf) >= self.ctx + 1:
                chunk = buf[:self.ctx + 1]
                buf = buf[self.ctx:]

                if skipped < self.skip:
                    skipped += self.ctx
                    continue

                yield (torch.tensor(chunk[:-1], dtype=torch.long),
                       torch.tensor(chunk[1:],  dtype=torch.long))
                yielded += self.ctx
                if yielded >= self.max_tokens:
                    return

class LegalDataset(IterableDataset):
    def __init__(self, local_train, tokenizer, cfg, max_tokens, skip_tokens=0):
        self.local_train = local_train
        self.tok = tokenizer
        self.ctx = cfg["ctx_len"]
        self.mix = cfg["mix"]
        self.max_tokens = max_tokens
        self.skip = skip_tokens

    def _stream_source(self, name, budget):
        path = self.local_train[name]
        yielded = 0
        buf = []
        with open(path, "rb") as f:
            for rec in ijson.items(f, "item"):
                text = rec.get("text", "")
                if len(text) < 100:
                    continue
                ids = self.tok.encode(text).ids
                buf.extend(ids)
                while len(buf) >= self.ctx + 1:
                    yield buf[:self.ctx + 1]
                    buf = buf[self.ctx:]
                    yielded += self.ctx
                    if yielded >= budget:
                        return

    def __iter__(self):
        sources = list(self.mix.keys())
        budgets = {s: int(self.max_tokens * r) for s, r in self.mix.items()}
        gens = {s: self._stream_source(s, budgets[s]) for s in sources}

        yielded = 0
        skipped = 0
        exhausted = set()

        while len(exhausted) < len(sources):
            for s in sources:
                if s in exhausted:
                    continue
                try:
                    chunk = next(gens[s])
                    if skipped < self.skip:
                        skipped += self.ctx
                        continue
                    yield (torch.tensor(chunk[:-1], dtype=torch.long),
                           torch.tensor(chunk[1:],  dtype=torch.long))
                    yielded += self.ctx
                    if yielded >= self.max_tokens:
                        return
                except StopIteration:
                    exhausted.add(s)

print("Streaming datasets initialized.")

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["wiki_lr"],
    betas=(CFG["beta1"], CFG["beta2"]),
    weight_decay=CFG["weight_decay"],
    fused=(DEVICE == "cuda")
)
scaler = torch.amp.GradScaler("cuda")

def get_lr(step, total_steps, lr_max, lr_min, warmup):
    if step < warmup:
        return lr_max * step / max(1, warmup)
    t = (step - warmup) / max(1, total_steps - warmup)
    return lr_min + (lr_max - lr_min) * 0.5 * (1 + math.cos(math.pi * min(1.0, t)))

def build_val_set(local_train, tokenizer, ctx_len, n_chunks=300):
    val_path = WORK / "val_chunks.pt"
    if val_path.exists():
        print("Loading cached validation set...")
        return torch.load(val_path)

    print("Building held-out validation set from Supreme Court judgments...")
    chunks, buf = [], []
    with open(local_train["train_supreme_court"], "rb") as f:
        for rec in ijson.items(f, "item"):
            text = rec.get("text", "")
            if len(text) < 100:
                continue
            buf.extend(tokenizer.encode(text).ids)
            while len(buf) >= ctx_len + 1 and len(chunks) < n_chunks:
                c = buf[:ctx_len + 1]
                chunks.append((torch.tensor(c[:-1]), torch.tensor(c[1:])))
                buf = buf[ctx_len:]
            if len(chunks) >= n_chunks:
                break

    torch.save(chunks, val_path)
    print(f"Validation set ready: {len(chunks)} sequences.")
    return chunks

@torch.no_grad()
def validate(model, val_chunks):
    get_raw_model(model).eval()
    losses = []
    for x, y in val_chunks:
        x, y = x.unsqueeze(0).to(DEVICE), y.unsqueeze(0).to(DEVICE)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _, loss = get_raw_model(model)(x, y)
        losses.append(loss.mean().item())
    get_raw_model(model).train()
    avg = sum(losses) / max(1, len(losses))
    return avg, math.exp(min(avg, 20.0))

val_chunks = build_val_set(local_train, tokenizer, CFG["ctx_len"])
print("Optimizer, lr scheduler, and validation pipeline ready.")

In [13]:
def session_elapsed_h():
    return (time.time() - SESSION_START) / 3600

if stage_done(1):
    print("Stage 1 already completed, restoring Stage 1 weights...")
    ckpt = torch.load(CKPT_DIR / "stage1_done.pt", map_location=DEVICE)
    get_raw_model(model).load_state_dict(ckpt["model"])
    print(f'Stage 1 loaded. val_loss={ckpt["val_loss"]:.4f}')
else:
    step, tokens, best_val = 0, 0, float("inf")
    ckpt_path = find_checkpoint(stage=1)
    if ckpt_path:
        step, tokens, best_val = load_checkpoint(ckpt_path, model, optimizer, scaler)
        print(f"Resuming Stage 1 from step {step} ({tokens/1e6:.1f}M tokens).")
    else:
        print("Starting Stage 1: Wikipedia Pre-training.")

    for pg in optimizer.param_groups:
        pg["lr"] = CFG["wiki_lr"]

    wiki_target = CFG["wiki_tokens"]
    total_steps = wiki_target // (CFG["batch_size"] * CFG["ctx_len"] * CFG["grad_accum"])
    stage_start = time.time()

    ds = WikiDataset(tokenizer, CFG["ctx_len"], wiki_target, skip_tokens=tokens)
    loader = DataLoader(ds, batch_size=CFG["batch_size"], num_workers=0, pin_memory=True)

    model.train()
    optimizer.zero_grad()
    micro = 0

    print(f'Stage 1 target: {wiki_target/1e6:.0f}M tokens (max {CFG["wiki_hours"]}h).')

    for x, y in loader:
        time_used = (time.time() - stage_start) / 3600
        if tokens >= wiki_target or time_used >= CFG["wiki_hours"]:
            print(f"Stage 1 ended: {tokens/1e6:.1f}M tokens processed in {time_used:.2f}h.")
            break

        x, y = x.to(DEVICE), y.to(DEVICE)
        lr = get_lr(step, total_steps, CFG["wiki_lr"], CFG["wiki_lr_min"], CFG["wiki_warmup"])
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        with torch.amp.autocast("cuda", dtype=torch.float16):
            _, loss = model(x, y)
        scaler.scale(loss.mean() / CFG["grad_accum"]).backward()
        micro += 1

        if micro % CFG["grad_accum"] != 0:
            continue

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        step += 1
        tokens += CFG["batch_size"] * CFG["ctx_len"] * CFG["grad_accum"]

        if step % CFG["log_every"] == 0:
            elapsed = time.time() - stage_start
            tok_s = tokens / max(1, elapsed)
            eta_h = (wiki_target - tokens) / max(1, tok_s) / 3600
            print(f"[Wiki] step={step:5d} | loss={loss.mean().item():.4f} | lr={lr:.2e} | "
                  f"tok={tokens/1e6:5.1f}M | speed={tok_s/1e3:.1f}K tok/s | ETA={eta_h:.2f}h")

        if step % CFG["save_every"] == 0:
            vl, vp = validate(model, val_chunks)
            save_checkpoint(1, step, tokens, vl, model, optimizer, scaler)
            best_val = min(best_val, vl)
            print(f"  [Val] loss={vl:.4f} | ppl={vp:.2f}")

    vl, vp = validate(model, val_chunks)
    mark_stage_done(1, model, vl, tokens)
    print(f"Stage 1 complete! Final val_loss={vl:.4f}, ppl={vp:.2f}. Total session={session_elapsed_h():.2f}h")

NameError: name 'stage_done' is not defined

[Wiki] step=  150 | loss=7.3248 | lr=4.47e-05 | tok= 19.7M | speed=9.9K tok/s | ETA=1.69h


[Wiki] step=  200 | loss=7.1049 | lr=5.97e-05 | tok= 26.2M | speed=9.9K tok/s | ETA=1.51h


[Wiki] step=  250 | loss=6.6049 | lr=7.47e-05 | tok= 32.8M | speed=9.9K tok/s | ETA=1.32h


[Wiki] step=  300 | loss=6.6025 | lr=8.97e-05 | tok= 39.3M | speed=9.9K tok/s | ETA=1.14h


[Wiki] step=  350 | loss=5.8222 | lr=1.05e-04 | tok= 45.9M | speed=9.9K tok/s | ETA=0.96h


... (2 outputs left out)

[Wiki] step=  500 | loss=5.4888 | lr=1.50e-04 | tok= 65.5M | speed=9.9K tok/s | ETA=0.41h


  Saved checkpoint: step=500 | tokens=65.5M | val_loss=6.1008
  [Val] loss=6.1008 | ppl=446.22


Stage 1 ended: 71.0M tokens processed in 2.00h.


Stage 1 successfully completed (val_loss=6.1372).
Stage 1 complete! Final val_loss=6.1372, ppl=462.76. Total session=2.03h


In [11]:
if stage_done(2):
    print("Stage 2 already completed, loading best model...")
    ckpt = torch.load(CKPT_DIR / "stage2_done.pt", map_location=DEVICE)
    get_raw_model(model).load_state_dict(ckpt["model"])
    print(f'Stage 2 loaded. val_loss={ckpt["val_loss"]:.4f}')
else:
    step, tokens, best_val = 0, 0, float("inf")
    ckpt_path = find_checkpoint(stage=2)
    if ckpt_path:
        step, tokens, best_val = load_checkpoint(ckpt_path, model, optimizer, scaler)
        print(f"Resuming Stage 2 from step {step} ({tokens/1e6:.1f}M tokens).")
    else:
        print("Starting Stage 2: Indian Legal Domain Adaptation.")

    for pg in optimizer.param_groups:
        pg["lr"] = CFG["legal_lr"]

    legal_target = CFG["legal_tokens"]
    total_steps = legal_target // (CFG["batch_size"] * CFG["ctx_len"] * CFG["grad_accum"])
    stage_start = time.time()

    ds = LegalDataset(local_train, tokenizer, CFG, legal_target, skip_tokens=tokens)
    loader = DataLoader(ds, batch_size=CFG["batch_size"], num_workers=0, pin_memory=True)

    model.train()
    optimizer.zero_grad()
    micro = 0

    print(f'Stage 2 target: {legal_target/1e6:.0f}M tokens (max {CFG["legal_hours"]}h).')

    for x, y in loader:
        time_used = (time.time() - stage_start) / 3600
        sess_used = session_elapsed_h()
        if tokens >= legal_target or time_used >= CFG["legal_hours"] or sess_used >= CFG["session_max_hours"]:
            print(f"Stage 2 ended: {tokens/1e6:.1f}M tokens in {time_used:.2f}h (session={sess_used:.2f}h).")
            break

        x, y = x.to(DEVICE), y.to(DEVICE)
        lr = get_lr(step, total_steps, CFG["legal_lr"], CFG["legal_lr_min"], CFG["legal_warmup"])
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        with torch.amp.autocast("cuda", dtype=torch.float16):
            _, loss = model(x, y)
        scaler.scale(loss.mean() / CFG["grad_accum"]).backward()
        micro += 1

        if micro % CFG["grad_accum"] != 0:
            continue

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        step += 1
        tokens += CFG["batch_size"] * CFG["ctx_len"] * CFG["grad_accum"]

        if step % CFG["log_every"] == 0:
            elapsed = time.time() - stage_start
            tok_s = tokens / max(1, elapsed)
            eta_h = (legal_target - tokens) / max(1, tok_s) / 3600
            print(f"[Legal] step={step:5d} | loss={loss.mean().item():.4f} | lr={lr:.2e} | "
                  f"tok={tokens/1e6:5.1f}M | speed={tok_s/1e3:.1f}K tok/s | ETA={eta_h:.2f}h")

        if step % CFG["save_every"] == 0:
            vl, vp = validate(model, val_chunks)
            save_checkpoint(2, step, tokens, vl, model, optimizer, scaler)
            if vl < best_val:
                best_val = vl
                torch.save({"model": get_raw_model(model).state_dict(),
                            "val_loss": vl}, CKPT_DIR / "best_model.pt")
                print(f"  ★ New best model saved! (val_loss={vl:.4f} | ppl={vp:.2f})")

    vl, vp = validate(model, val_chunks)
    mark_stage_done(2, model, vl, tokens)
    print(f"Stage 2 complete! Final val_loss={vl:.4f}, ppl={vp:.2f}. Total session={session_elapsed_h():.2f}h")

Starting Stage 2: Indian Legal Domain Adaptation.
Stage 2 target: 320M tokens (max 8.0h).


[Legal] step=   50 | loss=5.9834 | lr=9.80e-06 | tok=  6.6M | speed=9.9K tok/s | ETA=8.80h


[Legal] step=  100 | loss=5.6298 | lr=1.98e-05 | tok= 13.1M | speed=9.9K tok/s | ETA=8.63h


[Legal] step=  150 | loss=5.1202 | lr=2.98e-05 | tok= 19.7M | speed=9.9K tok/s | ETA=8.44h


[Legal] step=  200 | loss=4.9479 | lr=3.98e-05 | tok= 26.2M | speed=9.9K tok/s | ETA=8.26h


[Legal] step=  250 | loss=4.8267 | lr=4.98e-05 | tok= 32.8M | speed=9.9K tok/s | ETA=8.08h


... (44 outputs left out)

[Legal] step= 2100 | loss=2.8742 | lr=1.67e-05 | tok=275.3M | speed=9.8K tok/s | ETA=1.26h


[Legal] step= 2150 | loss=2.1492 | lr=1.49e-05 | tok=281.8M | speed=9.8K tok/s | ETA=1.08h


Stage 2 ended: 283.4M tokens in 8.00h (session=10.04h).


Stage 2 successfully completed (val_loss=3.6087).
Stage 2 complete! Final val_loss=3.6087, ppl=36.92. Total session=10.04h


In [19]:
from pathlib import Path
import os
import shutil
import time
import re
import torch

best_path = CKPT_DIR / "best_model.pt"
if not best_path.exists() and Path("/kaggle/input").exists():
    for f in Path("/kaggle/input").rglob("best_model.pt"):
        best_path = f
        print(f"Found model at: {best_path}")
        break
if Path("/kaggle/input").exists():
    for f in Path("/kaggle/input").rglob("vocab.json"):
        shutil.copy(f, TOK_DIR / "vocab.json")
    for f in Path("/kaggle/input").rglob("merges.txt"):
        shutil.copy(f, TOK_DIR / "merges.txt")

if best_path.exists():
    ckpt = torch.load(best_path, map_location=DEVICE)
    get_raw_model(model).load_state_dict(ckpt["model"])
    print(f'Loaded best model checkpoint (val_loss={ckpt["val_loss"]:.4f}).')
else:
    print("WARNING: best_model.pt not found! Using untrained current model weights.")
import gc
torch.cuda.empty_cache()
gc.collect()
raw_model = get_raw_model(model).to("cuda:0")
raw_model.eval()

with open(test_path, "r", encoding="utf-8") as f:
    test_dict = json.load(f)

test_items = list(test_dict.items())
total_test = len(test_items)
print(f'Generating continuations for {total_test} test prompts (batch_size={CFG["gen_batch"]})...')

predictions = {}
t_gen_start = time.time()
BATCH_SIZE = 8

for i in range(0, total_test, BATCH_SIZE):
    batch = test_items[i:i + BATCH_SIZE]
    batch_ids = [k for k, _ in batch]
    batch_prompts = [v["text"] for _, v in batch]

    tokenized_list = [tokenizer.encode(p).ids[-CFG["ctx_len"]:] for p in batch_prompts]
    max_len = max(len(t) for t in tokenized_list)

    padded_inputs = []
    for t in tokenized_list:
        pad_len = max_len - len(t)
        padded_inputs.append([0] * pad_len + t)

    inp_tensor = torch.tensor(padded_inputs, dtype=torch.long).to("cuda:0")
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out_tensor = raw_model.generate_batch(
            inp_tensor,
            max_new=CFG["max_new_tok"],
            temperature=CFG["temperature"],
            top_p=CFG["top_p"],
            rep_penalty=CFG["rep_penalty"]
        )

    for b_idx, eid in enumerate(batch_ids):
        gen_tokens = out_tensor[b_idx, max_len:].tolist()
        decoded = tokenizer.decode(gen_tokens)
        cleaned = re.sub(r"\s+", " ", decoded.replace("<s>", "").replace("</s>", "").replace("<unk>", "")).strip()
        words = cleaned.split()
        if len(words) >= CFG["target_words"]:
            continuation = " ".join(words[:CFG["target_words"]])
        elif len(words) > 0:
            continuation = cleaned
        else:

            continuation = " ".join(batch_prompts[b_idx].split()[-30:])
        predictions[eid] = {"text": continuation}

    done_cnt = min(i + BATCH_SIZE, total_test)
    sample_wc = len(predictions[batch_ids[0]]["text"].split())
    
    # Print progress for every single batch
    print(f"  Completed {done_cnt}/{total_test} ({100*done_cnt/total_test:.1f}%) | Last sample length: {sample_wc} words")
gen_elapsed = time.time() - t_gen_start
print(f"All {len(predictions)} continuations generated in {gen_elapsed/60:.2f} minutes!")

pred_path = WORK / "prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print(f"prediction.json successfully saved ({len(predictions)} entries).")

Found model at: /kaggle/input/notebooks/luciferk7/notebook49f05270cf/checkpoints/best_model.pt
Loaded best model checkpoint (val_loss=3.6280).
Generating continuations for 1000 test prompts (batch_size=16)...
DEBUG: Starting generation for 240 tokens. Batch shape: torch.Size([8, 647])
DEBUG: Step 0/240 took 0.1438 seconds.
DEBUG: Step 50/240 took 0.2739 seconds.
DEBUG: Step 100/240 took 0.3391 seconds.
DEBUG: Step 150/240 took 0.3427 seconds.
DEBUG: Step 200/240 took 0.3591 seconds.
  Completed 8/1000 (0.8%) | Last sample length: 128 words
DEBUG: Starting generation for 240 tokens. Batch shape: torch.Size([8, 698])
DEBUG: Step 0/240 took 0.0425 seconds.
DEBUG: Step 50/240 took 0.3290 seconds.
DEBUG: Step 100/240 took 0.3669 seconds.
DEBUG: Step 150/240 took 0.4032 seconds.
DEBUG: Step 200/240 took 0.4368 seconds.
  Completed 16/1000 (1.6%) | Last sample length: 128 words
DEBUG: Starting generation for 240 tokens. Batch shape: torch.Size([8, 739])
DEBUG: Step 0/240 took 0.0535 seconds.


In [22]:
zip_path = WORK / "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")

print("submission.zip successfully created!")
print("Checking archive structure:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in zf.namelist():
        print(f"  Archive entry: {name}")

print("\nSanity check on first 3 predictions:")
for eid in ["1", "2", "3"]:
    p = predictions[eid]["text"]
    wc = len(p.split())
    print(f"  ID {eid} ({wc} words): {p[:90]}...")

print("\nALL PHASES COMPLETE. Download submission.zip and submit to Codabench Phase 1!")

submission.zip successfully created!
Checking archive structure:
  Archive entry: prediction.json

Sanity check on first 3 predictions:
  ID 1 (128 words): not open for interference and therefore the High Court failed to consider or permit any fu...
  ID 2 (128 words): but no decision has been taken during trial. For that period of two weeks from today it ap...
  ID 3 (128 words): for doing so." 17. It is true that no sufficient ground to interfere with the said issue i...

ALL PHASES COMPLETE. Download submission.zip and submit to Codabench Phase 1!


In [23]:
from IPython.display import FileLink


display(FileLink('submission.zip'))

/kaggle/working/submission.zip